# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** 43  
**Kaggle challenge:** `Deep learning` (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "Choco Hunters"  

**Author 1 (sciper):** Ewa Miazga (367059)  
**Author 2 (sciper):** Sameh Lahouar (300454)   
**Author 3 (sciper):** Nour Guermazi (314474) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

## **01. Data and Data Augumentation**

## **02. Models**

## 02.a SSDLite + MobileNetV3 Architecture

The model is based on **SSDLite**, a lightweight variant of the Single Shot MultiBox Detector (SSD), paired with a **MobileNetV3** backbone. It processes input images and extracts multi-scale feature maps using depthwise separable convolutions and inverted residual blocks.

The **MobileNetV3 backbone** plays a crucial role in extracting meaningful features from the input image. It incorporates several efficiency-oriented design elements, such as **depthwise separable convolutions**, which significantly reduce the number of parameters and operations compared to standard convolutions. It also uses **inverted residual blocks** with skip connections to preserve useful information across layers, as well as **Squeeze-and-Excitation (SE) modules** to recalibrate feature channels adaptively. The **Hardswish activation function** - a lightweight approximation of swish—is also used to balance non-linearity and efficiency.

Once the backbone has processed the input image, the resulting feature maps are passed to a series of **auxiliary detection layers**. These layers operate at multiple spatial resolutions, allowing the model to detect objects of varying sizes. Unlike the original SSD, which uses standard convolutional layers for detection, **SSDLite replaces these with depthwise separable convolutions**, maintaining both speed and accuracy.

During inference, the model produces a set of predictions for each anchor box, including bounding box coordinates, class scores, and confidence values. These outputs are then filtered using **Non-Maximum Suppression (NMS)** and a confidence threshold to eliminate duplicate or low-confidence detections.

![SSDLite Architecture](images/ssdlite_architecture.png)

*Figure: SSDLite object detection pipeline with a MobileNet backbone and 320×320 input. The model extracts multi-scale features used for bounding box generation. This diagram shows MobileNetV2, which is conceptually similar to the MobileNetV3 backbone used in our implementation.*

Jelly Milk, Comtesse

### 🧪 Evaluation on Validation Set — Quantitative Results

The overall performance on the validation set was **moderate**, with an **F1-score of $0.81$**. However, two classes in particular — **Jelly Milk** and **Comtesse** — performed poorly. The **Comtesse** class likely suffers from visual ambiguity, as it is a white chocolate variety and often blends into the background due to similar coloration. In the case of **Jelly Milk**, the source of misclassification is less obvious and may stem from inconsistent visual patterns or limited representation in the training data.

Below are visual references for the two problematic classes:

<div align="center">
  <img src="images/Jelly_Milk.JPG" width="30%" style="margin: 0 50px;" />
  <img src="images/Comtesse.JPG" width="30%"  style="margin: 0 50px;" />
</div>

While the validation performance was acceptable, the **test set performance dropped to an F1-score of just $0.69$**, highlighting the model’s limited generalization ability. This suggests that while the current SSDLite-MobileNetV3 model is efficient, it may not have sufficient capacity to robustly capture complex visual differences across classes.

For context, the validation set consisted of $9$ samples, with $81$ used for training.
![SSDLite Evaluation](images/ssd-evaluation-res.png)
*Figure: Evaluation results of the SSDLite-MobileNetV3 model on the validation set.*

While the model performs strongly on several well-defined chocolate classes, its reliability drops significantly on more visually ambiguous ones. In particular, **Amandina** exhibits the weakest results, with an **F1-score of just $0.43$**. Out of $7$ samples, only $3$ were correctly classified, indicating that the model **struggles to learn or generalize this class**. This may be due to limited training data, high intra-class variability, or visual similarity to other chocolates.

Similarly, **Jelly Black**, despite having 10 correct predictions, shows a **noticeable confusion rate (~17%)**, suggesting that the model often confuses it with visually similar classes like **Jelly Milk** or **Tentation noir**. This highlights a broader limitation: the model is vulnerable to **misclassification among chocolates that share shape, texture, or color**, particularly in cluttered or poorly cropped image patches.

From this confusion matrix, we can expect the model to perform reliably on distinctive classes but to **struggle in edge cases** — especially when chocolates are visually similar or the detection input is imprecise. This underscores the importance of **robust detection and more discriminative features** for reliably separating closely related classes.


<p align="center">
  <img src="images/ssd_conf_matrix.png" width="60%" />
</p>
<p align="center"><em>Figure: Confusion Matrix</em></p>

### 🎯 Qualitative Results

The following examples show how the model performs qualitatively. When the model predicts a bounding box, it is generally accurate — as indicated by the **strong overlap between the predicted boxes (green)** and **the ground-truth boxes (red)**. This suggests that the confidence threshold was chosen appropriately, as predictions are only made when the model is reasonably certain.

However, in some cases, **no prediction is made for certain chocolates**, which could indicate that the threshold might be **too strict**, potentially filtering out uncertain but valid detections. Conversely, it could also suggest that the **confidence threshold is too low**, causing the model to suppress uncertain detections entirely.

Another key observation is that the model tends to **struggle in visually noisy environments** — for instance, when the background is cluttered, contains packaging, or shares visual similarity with the chocolate. This can significantly impact both detection accuracy and confidence.

<div align="center">
  <img src="images/ssd-eval-1.png" width="30%" style="margin: 0 10px;" />
  <img src="images/ssd-eval-2.png" width="30%" style="margin: 0 10px;" />
  <img src="images/ssd-eval-3.png" width="30%" style="margin: 0 10px;" />
</div>

*Figure 1: Ground-truth boxes (red) and predicted boxes (green)*

## 02.b MobileNetV3 + CNN Architecture

This two-stage pipeline combines **SSDLiteMobileNetV3** for detection and a custom **SimpleCNN** for classification. The architecture reuses the pretrained backbone from the detection model to extract patches of chocolates from full images. These patches are then passed individually to a lightweight CNN to classify the type of chocolate.


<div style="display: flex; justify-content: center; align-items: center; gap: 40px; margin: 20px 0;">
  <img src="images/cnn_architecture.png" width="30%" />
  <img src="images/ssd_cnn_architecture.png" width="30%" />
</div>

<p align="center"><em>Figure: SimpleCNN architecture (left) and SSDLite + CNN pipeline architecture (right).</em></p>

### 🧪 Evaluation on validation set - quantitive results

The performance of this model was noticeably **poor**, with an **F1-score of just $0.71$** on the validation set and a significant drop to **$0.42$ on the test set** — the **lowest among all tested models**.

**The bottleneck appears to lie in the SimpleCNN classifier**, rather than the MobileNetV3 detection backbone. The same backbone, when used in a previous SSDLite-only configuration, achieved considerably better results. While the SSDLite backbone performs well in detecting chocolates under controlled conditions, its performance tends to degrade in noisier environments — such as images with cluttered backgrounds, packaging, or overlapping objects.

Since the CNN operates **exclusively on the cropped patches provided by the detector**, its success is highly dependent on the **quality and accuracy of those detections**:

-	If the detector **misses a chocolate**, the CNN never has a chance to classify it.
-	If the detector **produces poorly cropped or noisy patches**, the CNN struggles to make a reliable prediction.

This architecture highlights the **limitations of a simple CNN classifier** when placed downstream of a real-world detection model. It serves as a useful probe into the **robustness of the classifier** under imperfect detection conditions.

Interestingly, the misclassified classes differ from those in the previous model. This time, the CNN struggles particularly with:

-	Tentation noir – F1-score: $0.36$
-	Jelly White – F1-score: $0.50$
-	Noir authentique – F1-score: $0.50$

![SSDLite Evaluation](images/cnn-evaluation-res.png)
*Figure: Evaluation results of the SSDLite-MobileNetV3 model on the validation set.*

While this model shows some improvement in overall recognition (e.g., higher diagonal intensity) compared to previous one, the confusion matrix reveals **clear weaknesses** that limit its reliability for fine-grained chocolate classification.

The most prominent issue appears with **Jelly White**, which has only **$5$ correct predictions out of $9$** and is **frequently confused with Jelly Milk ($4$ errors)**. This results in a **very low F1-score of 0.56**, highlighting the model’s difficulty in distinguishing between these two visually similar classes.

Another notable weakness is **Tentation noir**, with just **$4$ correct predictions and $4$ misclassifications**, evenly spread across **Jelly Milk, Jelly White, and Noir Authentique**. This suggests that the model **lacks strong discriminative features** to separate visually dark or structurally similar chocolates.

**Noir Authentique** also suffers from severe confusion — being misclassified as **Comtesse, Noblesse, and Tentation noir**, with no single dominant prediction class. The resulting **F1-score of $0.33$** reflects this ambiguity.

Even classes that performed well in previous models like **Stracciatella** ($0.50$ F1-score) and **Triangolo** ($0.67$ F1-score) now exhibit higher confusion, possibly due to **overfitting or increased class overlap introduced by augmented training**.

<p align="center">
  <img src="images/cnn_conf_matrix.png" width="60%" />
</p>
<p align="center"><em>Figure: Confusion Matrix</em></p>

## **03.c Faster R-CNN + MobileNetV3 with FPN**

This model was the one chosen for submission, it's executed directly in the main.py file and generates the csv file submitted on Kaggle.
The submitted versions runs only on an annotated version of the original training data and it yielded a score of $0.96951.$

The architecture is based on Faster R-CNN, a two-stage object detection framework, integrated with a lightweight MobileNetV3 backbone and a custom Feature Pyramid Network (FPN). This setup leverages both high-level semantics and fine spatial details by combining multi-scale feature maps extracted from intermediate layers of the backbone.

Specifically, layers $5$ and $9$ of the MobileNetV3-Large model are used to generate feature maps with $40$ and $80$ channels respectively. These are fed into an FPN to produce high-resolution features with uniform dimensionality (256 channels) for both the Region Proposal Network (RPN) and the Region of Interest (ROI) heads.

The RPN proposes candidate object regions using an anchor generator tailored with customized sizes and aspect ratios. These proposals are then classified and refined by the ROI heads, which consist of a two-layer MLP head followed by a class-specific bounding box regressor.

The model has been carefully balanced to remain under 12 million parameters ($10453044$ parameters to be exact), making it computationally efficient while maintaining competitive detection accuracy.

![Number of Parameters](images/NbParams.jpg)

*Figure: Detailed number of parameters of the model*

![FCNN structure](images/FCNN_structure.png)

*Figure: The custom Faster R-CNN pipeline using a MobileNetV3 backbone with FPN. Feature maps from intermediate layers are combined and fed to the RPN and ROI heads for region proposals and object classification.*


### 🧪 Evaluation on validation set - quantitive results

The dataset was split into 81 images for the training data and 9 images for the validation data. 

![Validation Predicition](images/ValPred.png)
*Figure: sample of the predictions on the validation dataset with the names of the classes written on the bounding boxes and the confidence scores*

![Qunatitavie results](images/QuantResultsFCNN.jpg)
*Figure: Overall and per class scores of the trained model on the validation dataset* 

<p align="center">
  <img src="images/mobile_conf_matrix_10M.png" width="60%" />
</p>
<p align="center"><em>Figure: Confusion Matrix</em></p>


The following graph shows the evolution of the validation loss as a function of the trained epochs, we can see that the value starts to converge around 100 epochs which may indicate that the training could stop there for similar results.
![Loss Plot](images/TrainingPlot.png)
*Figure: Plot of the Validation loss as a function of trained epochs* 


### 📈 Model Training on augmented dataset

We also managed to run our data on the augmented dataset described higher above, we chose not to submit is as our model since we can't upload our augmented data and since it only yielded a $0.97$ score which is very slightly higher than the score yielded with the normal data. The fact that the increase in the performance is not much higher can be explained with the fact that we only trained it on $100$ epochs(comparing to $150$ for the normal dataset) due to our lack of hardware for running and the expected consequent length of training( around $5$ hours more than for $100$ epochs)

Even if we didn't choose to submit this version of our model these are the scores calculated on the validation dataset$($27$ images out of $270$)

Here we can see that unlike the previous iteration of this model, $3$ classes don't have a perfect F1 score, this is explained by the fact that the validation dataset is bigger ($27$ images compaing to $9$) and thus its represents better the overall performance of prediction of our model.

![Validation prediction](images/ValPredAug.png)
*Figure: sample of the predictions on the validation dataset with the names of the classes written on the bounding boxes and the confidence scores* 

![Qunatitavie results](images/QuantResultsFCNNAug.jpg)
*Figure: Overall and per class scores of the trained model on the validation dataset from the augmented dataset* 

The following graph shows the evolution of the validation loss as a function of the trained epochs, here we can see that the validation loss was decreasing and could decrease even more with more training, which can explain why we didn't get the expected performance increase.

![Loss Plot](images/TrainingPlotAug.png)
*Figure: Plot of the Validation loss as a function of trained epochs* 


## 📊  **04. Results & Discussion**

<div align="center">
  <img src="images/Val-Test-Comp.png" width="45%" />
  <img src="images/Eval-Val-Dataset.png" width="45%" />
</div>

*Figure: F1-Score on Val and Test Set (Left) and Metrics about Val Set evaluation.(Right)*

<p align="center">
  <img src="images/per-1-class.png" width="60%" />
</p>

*Figure: F1-score per classification class evaluated on all the models* 


<div style="margin-top: 20px;"></div>

<u>1. F1-Score (Top Left)</u>

This graph compares the F1-score on both validation and test sets for all model variants.
Key observations:
-	The MobileNetV3 (8M and 10M) models perform the best, reaching F1-scores close to $1.0$, indicating excellent generalization.
-	The SSDLite + CNN combination shows the weakest performance, especially on the test set (F1 ≈ $0.42$), highlighting its sensitivity to noisy or imperfect detections.
-	Data augmentation further improves MobileNetV3 performance on the test set, closing the gap between training and real-world variation.

<div style="margin-top: 40px;"></div>


<u>2. Evaluation Metrics on Validation Set (Top Right)</u>

This bar chart provides a breakdown of loss, accuracy, and F1-score on the validation set.
Key insights:
-	CNN-based classification not only has the lowest F1-score but also the highest validation loss, confirming its instability.
-	All MobileNetV3-only variants (especially with 10M parameters and augmentation) yield low loss and high accuracy, reinforcing their robustness.
-	Interestingly, the SSDLite + MobileNetV3 combo has decent performance despite its smaller size, making it suitable when efficiency is prioritized over maximum accuracy.

<div style="margin-top: 40px;"></div>


<u>3. Per-Class F1-Score on Validation Set (Bottom)</u>

This graph reveals how each model performs across individual chocolate classes.
Highlights:
-	MobileNetV3 models (8M, 10M, 10M+augmented) show high and consistent F1-scores across all classes, including the previously problematic ones like Jelly Milk and Tentation noir.
-	In contrast, SSDLite + CNN exhibits uneven performance. Certain classes like Tentation noir, Jelly White, and Noir Authentique are poorly recognized, likely due to detection or patch quality issues.
-	SSDLite + MobileNetV3 shows a moderate performance spread, reinforcing its suitability as a balanced baseline model.

<div style="margin-top: 40px;"></div>

**<u>4. Conclusion</u>**

Based on the quantitative and qualitative analysis across all models, it becomes clear that **model architecture and capacity play a crucial role in performance**. The MobileNetV3 variants consistently outperform the simpler CNN-based classifier, both in overall metrics and per-class accuracy, demonstrating their ability to generalize effectively across diverse and visually complex samples. Notably, **MobileNetV3** (especially the 10M variant with augmentation) yielded results **that approach state-of-the-art (SOTA) performance for this task**, making it a reliable and scalable solution for chocolate multi-class classification. Additionally, **data augmentation proves to be a valuable strategy**, helping to close the performance gap between validation and test sets by making the model more robust to real-world variability. While the **SSDLite + MobileNetV3** combination offers a lightweight and efficient solution, its success is tightly coupled with the strength of the downstream classifier. This makes it a viable choice for deployment on constrained devices, but less reliable in challenging environments unless paired with a stronger recognition head.
